In [1]:
%matplotlib tk
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science','notebook', 'grid'])

In [74]:
N = 250

t, g = sp.symbols('t g')
m1, b = sp.symbols('m1 b')
L1, k1 = sp.symbols('L1 k1')

In [75]:
the1, r1 = sp.symbols(r'\theta_1, r_1',cls=sp.Function)

the1 = the1(t)
r1 = r1(t)

In [76]:
the1_d = sp.diff(the1, t)
r1_d = sp.diff(r1, t)
the1_dd = sp.diff(the1_d, t)
r1_dd = sp.diff(r1_d, t)

In [77]:
x1 = r1 * sp.sin(the1)
y1 = -r1 * sp.cos(the1)

In [78]:
# Kinetic Energy Term
T1 = 0.5 * m1 * (sp.diff(x1,t)**2+sp.diff(y1,t)**2)
T = T1

# Potential Energy Term
U1 = m1 * g * y1 + k1*(sp.sqrt(x1**2 + y1**2)-L1)**2/2
U = U1

# Lagrangian
L = (T - U)*sp.exp(b*t)

In [79]:
LE1 = sp.diff(L,the1) - sp.diff(sp.diff(L,the1_d),t).simplify()
LE2 = sp.diff(L,r1) - sp.diff(sp.diff(L,r1_d),t).simplify()

sols = sp.solve([LE1, LE2], (the1_dd, r1_dd), simplify = False, rational=False)

In [80]:
dw_1dt_f = sp.lambdify((t,g,m1,L1,k1,b,the1,r1,the1_d,r1_d), sols[the1_dd])
dv_1dt_f = sp.lambdify((t,g,m1,L1,k1,b,the1,r1,the1_d,r1_d), sols[r1_dd])
dthe1dt_f = sp.lambdify(the1_d,the1_d)
dr1dt_f = sp.lambdify(r1_d,r1_d)

In [81]:
def dSdt(S, t ,g, m1, L1, k1, b):
    the1, w1, r1, v1 = S
    return [
        dthe1dt_f(w1),
        dw_1dt_f(t, g, m1, L1, k1, b, the1, r1, w1, v1),
        dr1dt_f(v1),
        dv_1dt_f(t, g, m1, L1, k1, b, the1, r1, w1, v1)
    ]

In [98]:
t = np.linspace(0, 10, N)
g = 9.81
m1 = 2
L1 = 3.5
k1 = 25
b = 0.5
ans = sc.integrate.odeint(dSdt, y0=[0.5, 0, 5, 0], t=t, args=(g,m1,L1,k1, b))

In [99]:
the1 = ans.T[0]
r1 = ans.T[2]
plt.plot(t,the1)

In [100]:
def get_x1y1(t, the1,r1):
    return (
        r1 * np.sin(the1),
        -r1 * np.cos(the1),
    )

x1, y1 = get_x1y1(t, the1, r1)

In [101]:
def animate(i):
    ln1.set_data([0, x1[i]], [0, y1[i]])

In [102]:
from matplotlib import pyplot as plt, animation

fig, ax = plt.subplots()
ax.set_facecolor('k')
ax.set(xlim=(-4, 4), ylim=(-7, 1))
ln1, = plt.plot([], [], 'ro--', markersize=8)

ani = animation.FuncAnimation(fig, animate, frames = N-1, interval = 50)
#ani.save(filename="/Users/hasan/Python Animations/Double Pendulum.gif", writer="pillow")
#plt.show(ani)